# CERT Insider Threat — Weekly Feature Extraction Pipeline

**Architecture stateful / sans fuite temporelle**
- Chaque ligne = état d'un utilisateur à la fin d'une semaine W
- **Features locales** : agrégations brutes de la semaine W uniquement
- **Features stateful** : moyennes/std cumulées jusqu'à la fin de W (état du modèle)
- **Z-scores** : (valeur_W − moyenne_historique_{W-1,W-2,...}) / std_historique → 0 si < 3 semaines d'historique

**Garantie no-leakage** : `shift(1) + expanding()` → la semaine W ne voit jamais les données de W+1

In [ ]:
import pandas as pd
import numpy as np
import os, re, gc
import pyarrow.parquet as pq
from collections import defaultdict

pd.options.mode.chained_assignment = None  # silence SettingWithCopyWarning

In [ ]:
# ── Chemins ──────────────────────────────────────────────────────────────────
DATA_PATH   = 'cert_dataset'
OUTPUT_PATH = 'features_weekly.parquet'

# ── Constantes domaine ───────────────────────────────────────────────────────
DOMAIN = '@dtaa.com'
HOURS_OFF = (0, 7, 19, 24)   # < 7h ou > 19h = hors horaires
SHORT_SESSION_H  = 0.1        # sessions < 6 min = suspectes
SHORT_DEVICE_H   = 0.1
MIN_HIST_WEEKS   = 3          # nb minimum de semaines pour calculer un z-score fiable

# ── Mots-clés email ──────────────────────────────────────────────────────────
SUSPICIOUS_KEYWORDS = [
    'jobsearch','job search','security clearance','contingent upon award',
    'position requires','apply now','bachelor degree','job opening',
    'resignation','two weeks notice','last day','offer letter','new position',
    'keylogger','free keylogger','computer monitoring','monitoring software',
    'spyware','trojan','rootkit','RAT','remote access tool','payload','botnet',
    'do not share','do not forward','internal only','not for distribution',
    'keep this between','off the record',
    'my password is','login credentials','api key','private key','ssh key',
    'access token','secret key',
    'delete after reading','do not tell','just between us','they will never know',
]
SUSPICIOUS_EMAIL_PATTERN = '|'.join(map(re.escape, SUSPICIOUS_KEYWORDS))

# ── Domaines HTTP ─────────────────────────────────────────────────────────────
JOB_DOMAINS = {
    'linkedin.com','indeed.com','monster.com','simplyhired.com',
    'careerbuilder.com','glassdoor.com','job-hunt.org','jobhuntersbible.com',
    'taleo.net','elance.com','fiverr.com','freelancer.com','odesk.com',
    'ziprecruiter.com','dice.com','lever.co','greenhouse.io',
}
SUS_DOMAINS = {
    'wikileaks.org','dropbox.com','mediafire.com','megaupload.com',
    '4shared.com','filesonic.com','fileserve.com','yousendit.com',
    'refog.com','softactivity.com','thepiratebay.org','btjunkie.org',
    'torrentz.eu','demonoid.me','kat.ph','ilivid.com','filestube.com',
    'lockheedmartin.com','northropgrumman.com','boeing.com','raytheon.com',
    'harris.com','megaclick.com','backpage.com',
}
HTTP_WEIGHTS  = {'WWW Visit': 1, 'WWW Download': 2, 'WWW Upload': 4}
HTTP_KEYWORDS = [
    'resume','cv','cover letter','job offer','salary','offer letter',
    'interview','resignation','notice period','reference letter',
    'confidential','internal only','proprietary','trade secret',
    'do not distribute','restricted','source code','database dump',
    'credentials','password','api_key','secret_key','token',
    'drop table','delete all','rm -rf','wipe','disable account',
]
HTTP_KW_PATTERN = re.compile('|'.join(map(re.escape, HTTP_KEYWORDS)), flags=re.IGNORECASE)

# ── Helpers domaine HTTP ──────────────────────────────────────────────────────
def _extract_domain(url_series):
    return url_series.str.extract(r'https?://(?:www\.)?([^/?#]+)', expand=False).fillna('')

def _is_job_domain(url_series):
    patt = '|'.join(r'(?:^|\.)' + re.escape(d) + '$' for d in JOB_DOMAINS)
    return _extract_domain(url_series).str.contains(patt, regex=True, na=False)

def _is_sus_domain(url_series):
    patt = '|'.join(r'(?:^|\.)' + re.escape(d) + '$' for d in SUS_DOMAINS)
    return _extract_domain(url_series).str.contains(patt, regex=True, na=False)

# ── Utilitaire z-score sans look-ahead ───────────────────────────────────────
def zscore_vs_history(df, user_col, feature_col, min_periods=MIN_HIST_WEEKS):
    """
    Z-score de la semaine courante par rapport aux semaines précédentes UNIQUEMENT.
    Utilise shift(1) + expanding() → aucune fuite vers le futur.
    Renvoie une Series alignée sur df.
    """
    def _zscore_group(series):
        hist       = series.shift(1)
        hist_mean  = hist.expanding(min_periods=min_periods).mean()
        hist_std   = hist.expanding(min_periods=min_periods).std().clip(lower=1)
        return (series - hist_mean) / hist_std
    return df.groupby(user_col)[feature_col].transform(_zscore_group).fillna(0)

def cumulative_mean(df, user_col, feature_col):
    """Moyenne cumulée inclusive de la semaine W (état courant)."""
    return df.groupby(user_col)[feature_col].transform(
        lambda x: x.expanding(min_periods=1).mean()
    )

def cumulative_sum(df, user_col, feature_col):
    """Somme cumulée inclusive jusqu'à la semaine W."""
    return df.groupby(user_col)[feature_col].transform(
        lambda x: x.expanding(min_periods=1).sum()
    )

def safe_div(num, den, fill=0.0):
    """Division sécurisée (vectorisée)."""
    return np.where(den > 0, num / den, fill)

print('Config OK')

### Phase 1 — Chargement & preprocessing des données brutes

In [ ]:
# ── Users ──────────────────────────────────────────────────────────────────────
users = pd.read_parquet(f'{DATA_PATH}/users.parquet')
users['start_date'] = pd.to_datetime(users['start_date'])
users['end_date']   = pd.to_datetime(users['end_date'])
users.rename(columns={'user_id': 'user'}, inplace=True)

# Utilisateurs qui ont quitté avant la fin du dataset (2011-06-01)
DATASET_END   = pd.Timestamp('2011-06-01')
DATASET_START = pd.Timestamp('2010-01-04')  # premier lundi
users['has_left'] = users['end_date'] < DATASET_END

# Dictionnaire end_date par utilisateur (utile pour les features post-départ)
user_end_date = dict(zip(users['user'], users['end_date']))

print(f'{len(users)} utilisateurs — {users["has_left"].sum()} ont quitté avant la fin')

In [ ]:
# ── Email : chargement & flags ────────────────────────────────────────────────
email = pd.read_parquet(f'{DATA_PATH}/email.parquet')
email['date'] = pd.to_datetime(email['date'], format='%m/%d/%Y %H:%M:%S')
email = email.drop(columns=['id']).sort_values('date')

# Flags événementiels
email['is_external']              = (~email['from'].str.contains(DOMAIN, na=False)) | (~email['to'].str.contains(DOMAIN, na=False))
email['has_bcc']                  = email['bcc'].notna() & (email['bcc'] != 'None')
email['has_attachment']           = email['attachments'].notna() & (email['attachments'] != 'None')
email['is_external_with_att']     = email['is_external'] & email['has_attachment']
email['is_after_hours']           = (email['date'].dt.hour > 19) | (email['date'].dt.hour < 7)
email['is_weekend']               = email['date'].dt.dayofweek >= 5
email['is_off_hours_or_weekend']  = email['is_after_hours'] | email['is_weekend']
email['has_suspicious_content']   = email['content'].str.lower().str.contains(SUSPICIOUS_EMAIL_PATTERN, na=False).astype(int)

# Granularité temporelle
email['week_start'] = email['date'].dt.to_period('W').dt.start_time

# PC le plus utilisé par utilisateur (référence globale, stable)
email_main_pc = (
    email.groupby(['user', 'pc']).size()
    .reset_index(name='cnt').sort_values('cnt', ascending=False)
    .groupby('user').first().reset_index()[['user', 'pc']]
    .rename(columns={'pc': 'email_main_pc'})
)
email = email.merge(email_main_pc, on='user', how='left')
email['is_not_main_pc'] = (email['pc'] != email['email_main_pc']).astype(int)

print(f'Email : {len(email):,} événements')

In [ ]:
# ── Logon : chargement & sessions ────────────────────────────────────────────
logon = pd.read_parquet(f'{DATA_PATH}/logon.parquet')
logon['date'] = pd.to_datetime(logon['date'], format='%m/%d/%Y %H:%M:%S')
logon = logon.drop(columns=['id']).sort_values('date')

logon['is_after_hours'] = (logon['date'].dt.hour > 19) | (logon['date'].dt.hour < 7)
logon['week_start']     = logon['date'].dt.to_period('W').dt.start_time

# PC principal par utilisateur
logon_main_pc = (
    logon[logon['activity']=='Logon']
    .groupby(['user','pc']).size()
    .reset_index(name='cnt').sort_values('cnt',ascending=False)
    .groupby('user').first().reset_index()[['user','pc']]
    .rename(columns={'pc':'logon_main_pc'})
)
logon = logon.merge(logon_main_pc, on='user', how='left')
logon['is_not_main_pc'] = (logon['pc'] != logon['logon_main_pc']).astype(int)

# ── Construction des sessions Logon→Logoff par rang ──────────────────────────
logon_ev  = logon[logon['activity']=='Logon'].copy()
logoff_ev = logon[logon['activity']=='Logoff'].copy()
logon_ev['rank']  = logon_ev.groupby('user').cumcount()
logoff_ev['rank'] = logoff_ev.groupby('user').cumcount()

sessions = pd.merge(
    logon_ev[['user','rank','pc','date','week_start','is_after_hours']],
    logoff_ev[['user','rank','date']].rename(columns={'date':'date_end'}),
    on=['user','rank'], how='inner'
)
sessions.rename(columns={'date': 'date_start'}, inplace=True)
sessions['duration_h'] = (sessions['date_end'] - sessions['date_start']).dt.total_seconds() / 3600
sessions = sessions[(sessions['duration_h'] > 0) & (sessions['duration_h'] < 24)]
sessions['is_short'] = sessions['duration_h'] < SHORT_SESSION_H

# Multi-PC simultané : deux Logon consécutifs sur des PCs différents
logon_ev['prev_pc'] = logon_ev.groupby('user')['pc'].shift(1)
logon_ev['prev_act'] = logon_ev.groupby('user')['activity'].shift(1)
logon_ev['multi_pc_flag'] = (
    (logon_ev['activity']=='Logon') &
    (logon_ev['prev_act']=='Logon') &
    (logon_ev['pc'] != logon_ev['prev_pc'])
).astype(int)

print(f'Logon : {len(logon):,} événements | Sessions valides : {len(sessions):,}')

In [ ]:
# ── File : chargement & flags ─────────────────────────────────────────────────
file = pd.read_parquet(f'{DATA_PATH}/file.parquet')
file['date'] = pd.to_datetime(file['date'], format='%m/%d/%Y %H:%M:%S')
file = file.drop(columns=['id']).sort_values('date')

# Flags d'activité
file['is_copy']      = (file['activity'] == 'Copy').astype(int)
file['is_write']     = (file['activity'] == 'Write').astype(int)
file['is_delete']    = (file['activity'] == 'Delete').astype(int)
file['is_open']      = (file['activity'] == 'Open').astype(int)
file['copy_to_rem']  = ((file['activity']=='Copy') & file['to_removable_media'].fillna(False)).astype(int)
file['from_rem']     = ((file['activity'].isin(['Copy','Open'])) & file['from_removable_media'].fillna(False)).astype(int)
file['is_off_hours'] = ((file['date'].dt.hour > 19) | (file['date'].dt.hour < 7) | (file['date'].dt.dayofweek >= 5)).astype(int)

# Contenu suspect dans le nom de fichier
file_sus_ext = r'\.(exe|bat|ps1|sh|zip|7z|rar|tar|gz|dll|sys)$'
file['is_suspicious'] = file['filename'].str.lower().str.contains(file_sus_ext, na=False).astype(int)

# Flags séquentiels (au niveau événement) — opère sur l'historique trié
file['prev_act']  = file.groupby(['user','filename'])['activity'].shift(1)
file['open_then_copy']    = ((file['activity']=='Copy')   & (file['prev_act']=='Open')).astype(int)
file['write_then_delete'] = ((file['activity']=='Delete') & (file['prev_act']=='Write')).astype(int)
file['copy_then_delete']  = ((file['activity']=='Delete') & (file['prev_act']=='Copy')).astype(int)

file['week_start'] = file['date'].dt.to_period('W').dt.start_time

# Decoy files
decoy = pd.read_parquet(f'{DATA_PATH}/decoy.parquet')  # columns: filename, pc
decoy_filenames = set(decoy['filename'].str.lower())
file['is_decoy'] = file['filename'].str.lower().isin(decoy_filenames).astype(int)

# PC principal
file_main_pc = (
    file.groupby(['user','pc']).size().reset_index(name='cnt').sort_values('cnt',ascending=False)
    .groupby('user').first().reset_index()[['user','pc']].rename(columns={'pc':'file_main_pc'})
)
file = file.merge(file_main_pc, on='user', how='left')
file['is_not_main_pc'] = (file['pc'] != file['file_main_pc']).astype(int)

print(f'File : {len(file):,} événements')

In [ ]:
# ── Device : chargement & intervalles ────────────────────────────────────────
device = pd.read_parquet(f'{DATA_PATH}/device.parquet')
device['date'] = pd.to_datetime(device['date'], format='%m/%d/%Y %H:%M:%S')
device = device.drop(columns=['id']).sort_values('date')

device['is_after_hours'] = ((device['date'].dt.hour > 19) | (device['date'].dt.hour < 7)).astype(int)
device['week_start']     = device['date'].dt.to_period('W').dt.start_time

# Connexions uniquement ("Connect")
device_con = device[device['activity'] == 'Connect'].copy()
device_dis = device[device['activity'] == 'Disconnect'].copy()

device_con['rank'] = device_con.groupby('user').cumcount()
device_dis['rank'] = device_dis.groupby('user').cumcount()

device_sessions = pd.merge(
    device_con[['user','rank','pc','date','week_start','is_after_hours']],
    device_dis[['user','rank','date']].rename(columns={'date':'date_end'}),
    on=['user','rank'], how='inner'
)
device_sessions.rename(columns={'date':'date_start'}, inplace=True)
device_sessions['duration_h'] = (device_sessions['date_end'] - device_sessions['date_start']).dt.total_seconds() / 3600
device_sessions = device_sessions[(device_sessions['duration_h'] >= 0) & (device_sessions['duration_h'] < 48)]
device_sessions['is_short'] = (device_sessions['duration_h'] < SHORT_DEVICE_H).astype(int)

# PC principal
device_main_pc = (
    device_con.groupby(['user','pc']).size().reset_index(name='cnt').sort_values('cnt',ascending=False)
    .groupby('user').first().reset_index()[['user','pc']].rename(columns={'pc':'device_main_pc'})
)
device_con = device_con.merge(device_main_pc, on='user', how='left')
device_con['is_not_main_pc'] = (device_con['pc'] != device_con['device_main_pc']).astype(int)

print(f'Device : {len(device_con):,} connexions | Sessions valides : {len(device_sessions):,}')

In [ ]:
# ── Phase 2 : Grille hebdomadaire complète (user × semaine) ───────────────────
# Chaque utilisateur a une ligne pour chaque semaine entre son arrivée et sa sortie.
# Les semaines sans activité seront remplies avec 0.

all_mondays = pd.date_range(start=DATASET_START, end=DATASET_END, freq='W-MON')

user_week_frames = []
for _, row in users.iterrows():
    # Restreindre la période aux semaines où l'utilisateur était actif
    u_start = max(DATASET_START, row['start_date'].to_period('W').start_time)
    u_end   = min(DATASET_END,   row['end_date'].to_period('W').start_time + pd.Timedelta(weeks=4))  # légère marge post-départ
    weeks_i = all_mondays[(all_mondays >= u_start) & (all_mondays <= u_end)]
    user_week_frames.append(
        pd.DataFrame({'user': row['user'], 'week_start': weeks_i})
    )

weekly_grid = pd.concat(user_week_frames, ignore_index=True)
print(f'Grille hebdomadaire : {len(weekly_grid):,} lignes ({len(users)} utilisateurs × ~{len(all_mondays)} semaines)')

### Phase 3 — Email features hebdomadaires

In [ ]:
# ── Agrégation brute par (user, semaine) ──────────────────────────────────────
email_w = email.groupby(['user','week_start']).agg(
    nb_emails         = ('date',                       'count'),
    nb_sent           = ('activity',                   lambda x: (x=='Send').sum()),
    nb_external       = ('is_external',                'sum'),
    nb_bcc            = ('has_bcc',                    'sum'),
    nb_ext_att        = ('is_external_with_att',       'sum'),
    nb_off_hrs        = ('is_off_hours_or_weekend',    'sum'),
    nb_suspicious     = ('has_suspicious_content',     'sum'),
    nb_not_main_pc    = ('is_not_main_pc',             'sum'),
).reset_index()

# Taille moyenne des emails envoyés cette semaine
sent_size_w = (
    email[email['activity']=='Send']
    .groupby(['user','week_start'])['size'].mean()
    .reset_index(name='avg_size_sent')
)
email_w = email_w.merge(sent_size_w, on=['user','week_start'], how='left')
email_w['avg_size_sent'] = email_w['avg_size_sent'].fillna(0)

# ── Merge avec la grille & remplissage des semaines sans activité ─────────────
email_w = weekly_grid.merge(email_w, on=['user','week_start'], how='left').fillna(0)
email_w = email_w.sort_values(['user','week_start'])

# ── Features locales (ratios de la semaine W uniquement) ──────────────────────
n_e = email_w['nb_emails'].clip(lower=1)  # dénominateur sécurisé
email_w['email_external_ratio_w']     = email_w['nb_external']    / n_e
email_w['email_suspicious_ratio_w']   = email_w['nb_suspicious']  / n_e
email_w['email_bcc_ratio_w']          = email_w['nb_bcc']         / n_e
email_w['email_ext_att_ratio_w']      = email_w['nb_ext_att']     / n_e
email_w['email_off_hrs_ratio_w']      = email_w['nb_off_hrs']     / n_e
email_w['email_not_main_pc_ratio_w']  = email_w['nb_not_main_pc'] / n_e

# ── Features stateful : moyenne cumulée des emails/semaine jusqu'à W ──────────
email_w['email_cumul_avg_per_week']   = cumulative_mean(email_w, 'user', 'nb_emails')
email_w['email_cumul_avg_size_sent']  = cumulative_mean(email_w, 'user', 'avg_size_sent')

# ── Z-scores : déviation de la semaine W par rapport à l'historique W-1,W-2,… ─
email_w['email_zscore_nb_w']      = zscore_vs_history(email_w, 'user', 'nb_emails')
email_w['email_zscore_sent_w']    = zscore_vs_history(email_w, 'user', 'nb_sent')
email_w['email_zscore_ext_w']     = zscore_vs_history(email_w, 'user', 'nb_external')
email_w['email_zscore_sus_w']     = zscore_vs_history(email_w, 'user', 'nb_suspicious')

print(f'Email hebdo : {email_w.shape}')

### Phase 3 — Logon features hebdomadaires

In [ ]:
# ── Agrégation brute des événements de connexion ──────────────────────────────
logon_w = logon.groupby(['user','week_start']).agg(
    nb_logon           = ('activity', lambda x: (x=='Logon').sum()),
    nb_logoff          = ('activity', lambda x: (x=='Logoff').sum()),
    nb_after_hrs       = ('is_after_hours', 'sum'),
    nb_pc_distinct     = ('pc', 'nunique'),
    nb_not_main_pc     = ('is_not_main_pc', 'sum'),
).reset_index()

# Multi-PC simultané hebdomadaire
multi_pc_w = (
    logon_ev.groupby(['user','week_start'])['multi_pc_flag'].sum()
    .reset_index(name='nb_multi_pc_logon')
)
logon_w = logon_w.merge(multi_pc_w, on=['user','week_start'], how='left')

# ── Agrégation des sessions (durée, sessions courtes) ─────────────────────────
sessions_w = sessions.groupby(['user','week_start']).agg(
    session_duration_avg = ('duration_h', 'mean'),
    session_duration_std = ('duration_h', 'std'),
    session_nb           = ('duration_h', 'count'),
    session_nb_short     = ('is_short',    'sum'),
).reset_index()

logon_w = logon_w.merge(sessions_w, on=['user','week_start'], how='left')

# ── Merge grille & fill ───────────────────────────────────────────────────────
logon_w = weekly_grid.merge(logon_w, on=['user','week_start'], how='left').fillna(0)
logon_w = logon_w.sort_values(['user','week_start'])

# ── Features locales ──────────────────────────────────────────────────────────
n_l = logon_w['nb_logon'].clip(lower=1)
n_s = logon_w['session_nb'].clip(lower=1)

logon_w['logon_unclosed_ratio_w']       = (logon_w['nb_logon'] - logon_w['nb_logoff']).clip(lower=0) / n_l
logon_w['logon_after_hrs_ratio_w']      = logon_w['nb_after_hrs']   / n_l
logon_w['logon_multi_pc_flag_w']        = (logon_w['nb_multi_pc_logon'] > 0).astype(int)
logon_w['logon_short_session_ratio_w']  = logon_w['session_nb_short'] / n_s
logon_w['logon_not_main_pc_ratio_w']    = logon_w['nb_not_main_pc']  / n_l

# ── Features stateful ────────────────────────────────────────────────────────
logon_w['logon_cumul_avg_session_h']   = cumulative_mean(logon_w, 'user', 'session_duration_avg')
logon_w['logon_cumul_avg_per_week']    = cumulative_mean(logon_w, 'user', 'nb_logon')

# ── Z-scores ──────────────────────────────────────────────────────────────────
logon_w['logon_zscore_nb_w']          = zscore_vs_history(logon_w, 'user', 'nb_logon')
logon_w['logon_zscore_after_hrs_w']   = zscore_vs_history(logon_w, 'user', 'nb_after_hrs')
logon_w['logon_zscore_session_dur_w'] = zscore_vs_history(logon_w, 'user', 'session_duration_avg')

print(f'Logon hebdo : {logon_w.shape}')

### Phase 3 — File features hebdomadaires

In [ ]:
# ── Agrégation brute ──────────────────────────────────────────────────────────
file_w = file.groupby(['user','week_start']).agg(
    nb_events          = ('date',               'count'),
    nb_copy            = ('is_copy',            'sum'),
    nb_write           = ('is_write',           'sum'),
    nb_delete          = ('is_delete',          'sum'),
    nb_copy_to_rem     = ('copy_to_rem',        'sum'),
    nb_from_rem        = ('from_rem',           'sum'),
    nb_unique_files    = ('filename',           'nunique'),
    nb_off_hrs         = ('is_off_hours',       'sum'),
    nb_suspicious      = ('is_suspicious',      'sum'),
    nb_open_then_copy  = ('open_then_copy',     'sum'),
    nb_write_del       = ('write_then_delete',  'sum'),
    nb_copy_del        = ('copy_then_delete',   'sum'),
    nb_decoy           = ('is_decoy',           'sum'),
    nb_not_main_pc     = ('is_not_main_pc',     'sum'),
).reset_index()

# ── Merge grille & fill ───────────────────────────────────────────────────────
file_w = weekly_grid.merge(file_w, on=['user','week_start'], how='left').fillna(0)
file_w = file_w.sort_values(['user','week_start'])

# ── Features locales (ratios de la semaine W) ─────────────────────────────────
n_ev   = file_w['nb_events'].clip(lower=1)
n_cp   = file_w['nb_copy'].clip(lower=1)
n_wr   = file_w['nb_write'].clip(lower=1)
n_fi   = file_w['nb_unique_files'].clip(lower=1)

file_w['file_copy_ratio_w']           = file_w['nb_copy']           / n_ev
file_w['file_write_ratio_w']          = file_w['nb_write']          / n_ev
file_w['file_delete_ratio_w']         = file_w['nb_delete']         / n_ev
file_w['file_copy_to_rem_ratio_w']    = file_w['nb_copy_to_rem']    / n_cp
file_w['file_from_rem_ratio_w']       = file_w['nb_from_rem']       / n_ev
file_w['file_events_per_file_w']      = file_w['nb_events']         / n_fi
file_w['file_off_hrs_ratio_w']        = file_w['nb_off_hrs']        / n_ev
file_w['file_suspicious_ratio_w']     = file_w['nb_suspicious']     / n_ev
file_w['file_open_then_copy_ratio_w'] = file_w['nb_open_then_copy'] / n_cp
file_w['file_copy_then_del_ratio_w']  = file_w['nb_copy_del']       / n_cp
file_w['file_write_then_del_ratio_w'] = file_w['nb_write_del']      / n_wr
file_w['file_not_main_pc_ratio_w']    = file_w['nb_not_main_pc']    / n_ev

# Accès decoy cumulatif (signal fort : un seul accès est déjà suspect)
file_w['decoy_cumul_access']          = cumulative_sum(file_w, 'user', 'nb_decoy')

# ── Features stateful ────────────────────────────────────────────────────────
file_w['file_cumul_avg_per_week']     = cumulative_mean(file_w, 'user', 'nb_events')

# ── Z-scores ──────────────────────────────────────────────────────────────────
file_w['file_zscore_nb_w']            = zscore_vs_history(file_w, 'user', 'nb_events')
file_w['file_zscore_copy_w']          = zscore_vs_history(file_w, 'user', 'nb_copy')
file_w['file_zscore_rem_w']           = zscore_vs_history(file_w, 'user', 'nb_copy_to_rem')

print(f'File hebdo : {file_w.shape}')

### Phase 3 — Device features hebdomadaires

In [ ]:
# ── Agrégation brute des connexions ──────────────────────────────────────────
device_w = device_con.groupby(['user','week_start']).agg(
    nb_conn        = ('date',            'count'),
    nb_after_hrs   = ('is_after_hours',  'sum'),
    nb_not_main_pc = ('is_not_main_pc',  'sum'),
).reset_index()

# Durée et ratio courte connexion (via sessions device)
dev_sess_w = device_sessions.groupby(['user','week_start']).agg(
    con_duration_avg = ('duration_h', 'mean'),
    con_duration_std = ('duration_h', 'std'),
    nb_short_conn    = ('is_short',    'sum'),
    nb_sessions      = ('duration_h',  'count'),
).reset_index()

device_w = device_w.merge(dev_sess_w, on=['user','week_start'], how='left')

# ── Merge grille & fill ───────────────────────────────────────────────────────
device_w = weekly_grid.merge(device_w, on=['user','week_start'], how='left').fillna(0)
device_w = device_w.sort_values(['user','week_start'])

# ── Features locales ──────────────────────────────────────────────────────────
n_c  = device_w['nb_conn'].clip(lower=1)
n_cs = device_w['nb_sessions'].clip(lower=1)

device_w['device_after_hrs_ratio_w']  = device_w['nb_after_hrs']   / n_c
device_w['device_short_conn_ratio_w'] = device_w['nb_short_conn']  / n_cs
device_w['device_not_main_pc_ratio_w']= device_w['nb_not_main_pc'] / n_c

# ── Features stateful ────────────────────────────────────────────────────────
device_w['device_cumul_avg_per_week']     = cumulative_mean(device_w, 'user', 'nb_conn')
device_w['device_cumul_avg_duration_h']   = cumulative_mean(device_w, 'user', 'con_duration_avg')

# ── Z-scores ──────────────────────────────────────────────────────────────────
device_w['device_zscore_nb_w']        = zscore_vs_history(device_w, 'user', 'nb_conn')
device_w['device_zscore_after_hrs_w'] = zscore_vs_history(device_w, 'user', 'nb_after_hrs')

print(f'Device hebdo : {device_w.shape}')

### Phase 3 — HTTP features hebdomadaires (traitement par chunks)

In [ ]:
# ── Accumulation (user, semaine) en mémoire contrôlée ────────────────────────
# Chaque batch de 1M lignes est traité vectoriellement.
# Les domaines uniques sont collectés séparément pour éviter les doublons inter-chunks.

http_chunks    = []    # liste de DataFrames agrégés par (user, semaine)
http_dom_chunks = []   # liste de (user, week, domain) pour comptage unique

pf = pq.ParquetFile(f'{DATA_PATH}/http.parquet')

for i, batch in enumerate(pf.iter_batches(batch_size=1_000_000)):
    chunk = batch.to_pandas()
    chunk['activity']    = chunk['activity'].astype(str)
    chunk['datetime']    = pd.to_datetime(chunk['date'])
    chunk['week_start']  = chunk['datetime'].dt.to_period('W').dt.start_time
    chunk['hour']        = chunk['datetime'].dt.hour
    chunk['dow']         = chunk['datetime'].dt.dayofweek
    chunk['day']         = chunk['datetime'].dt.date

    # Flags
    chunk['is_off_hrs']  = ((chunk['dow'] < 5) & ((chunk['hour'] < 7) | (chunk['hour'] >= 19))).astype(int)
    chunk['is_weekend']  = (chunk['dow'] >= 5).astype(int)
    chunk['is_off_or_wk']= (chunk['is_off_hrs'] | chunk['is_weekend']).astype(int)
    chunk['weight']      = chunk['activity'].map(HTTP_WEIGHTS).fillna(0)
    chunk['is_upload']   = (chunk['activity'] == 'WWW Upload').astype(int)
    chunk['is_download'] = (chunk['activity'] == 'WWW Download').astype(int)
    chunk['is_job']      = _is_job_domain(chunk['url']).astype(int)
    chunk['is_sus']      = _is_sus_domain(chunk['url']).astype(int)
    chunk['job_w']       = chunk['weight'] * chunk['is_job']
    chunk['_domain']     = _extract_domain(chunk['url'])

    # Keywords : uniquement sur uploads/downloads
    transfer_mask = chunk['activity'].isin(['WWW Upload','WWW Download'])
    chunk.loc[transfer_mask, 'kw_flag'] = (
        chunk.loc[transfer_mask,'content'].fillna('').str.count(HTTP_KW_PATTERN)
    )
    chunk['kw_flag'] = chunk['kw_flag'].fillna(0).astype(int)

    # Agrégation vectorisée par (user, semaine)
    agg = chunk.groupby(['user','week_start']).agg(
        nb_total       = ('datetime',    'count'),
        nb_upload      = ('is_upload',   'sum'),
        nb_download    = ('is_download', 'sum'),
        nb_off_or_wk   = ('is_off_or_wk','sum'),
        nb_kw_flag     = ('kw_flag',     'sum'),
        nb_sus_domain  = ('is_sus',      'sum'),
        total_weighted = ('weight',      'sum'),
        job_weighted   = ('job_w',       'sum'),
        nb_days_active = ('day',         'nunique'),
    ).reset_index()
    http_chunks.append(agg)

    # Domaines uniques : (user, week, domain) dédupliqué par chunk
    dom_chunk = chunk[['user','week_start','_domain']].drop_duplicates()
    http_dom_chunks.append(dom_chunk)

    del chunk, agg, dom_chunk, transfer_mask
    gc.collect()

    if i % 10 == 0:
        print(f'Batch {i} — {(i+1)*1_000_000:,} lignes traitées')

print('\nAgrégation finale HTTP...')

# Combiner tous les chunks et ré-agréger (car un (user, week) peut être à cheval sur 2 chunks)
http_raw = (
    pd.concat(http_chunks, ignore_index=True)
    .groupby(['user','week_start'])
    .agg({
        'nb_total':       'sum',
        'nb_upload':      'sum',
        'nb_download':    'sum',
        'nb_off_or_wk':   'sum',
        'nb_kw_flag':     'sum',
        'nb_sus_domain':  'sum',
        'total_weighted': 'sum',
        'job_weighted':   'sum',
        'nb_days_active': 'sum',  # approximation (borne supérieure, jours peuvent se chevaucher)
    })
    .reset_index()
)

# Domaines uniques par (user, semaine) — déduplication globale
http_unique_domains = (
    pd.concat(http_dom_chunks, ignore_index=True)
    .drop_duplicates(subset=['user','week_start','_domain'])
    .groupby(['user','week_start'])['_domain'].nunique()
    .reset_index(name='nb_unique_domains')
)
http_raw = http_raw.merge(http_unique_domains, on=['user','week_start'], how='left')

del http_chunks, http_dom_chunks
gc.collect()
print(f'HTTP brut : {len(http_raw):,} (user, semaine) paires')

In [ ]:
# ── Merge grille & fill ───────────────────────────────────────────────────────
http_w = weekly_grid.merge(http_raw, on=['user','week_start'], how='left').fillna(0)
http_w = http_w.sort_values(['user','week_start'])

# ── Features locales ──────────────────────────────────────────────────────────
n_h  = http_w['nb_total'].clip(lower=1)
n_tw = http_w['total_weighted'].clip(lower=1)
n_d  = http_w['nb_days_active'].clip(lower=1)

http_w['http_upload_ratio_w']       = http_w['nb_upload']       / n_h
http_w['http_download_ratio_w']     = http_w['nb_download']      / n_h
http_w['http_off_hrs_ratio_w']      = http_w['nb_off_or_wk']     / n_h
http_w['http_sus_domain_ratio_w']   = http_w['nb_sus_domain']    / n_h
http_w['http_job_search_score_w']   = http_w['job_weighted']     / n_tw
http_w['http_avg_req_per_day_w']    = http_w['nb_total']         / n_d
http_w['http_unique_domains_w']     = http_w['nb_unique_domains']
http_w['http_kw_flags_w']           = http_w['nb_kw_flag']

# ── Features stateful ────────────────────────────────────────────────────────
http_w['http_cumul_avg_per_week']   = cumulative_mean(http_w, 'user', 'nb_total')

# ── Z-scores ──────────────────────────────────────────────────────────────────
http_w['http_zscore_nb_w']          = zscore_vs_history(http_w, 'user', 'nb_total')
http_w['http_zscore_upload_w']      = zscore_vs_history(http_w, 'user', 'nb_upload')
http_w['http_zscore_kw_w']          = zscore_vs_history(http_w, 'user', 'nb_kw_flag')

print(f'HTTP hebdo : {http_w.shape}')

### Phase 3 — Features statiques & LDAP

In [ ]:
# ── Psychométrique (OCEAN) — constante par utilisateur ───────────────────────
psycho = pd.read_parquet(f'{DATA_PATH}/psychometric.parquet')
psycho.rename(columns={'user_id':'user'}, inplace=True)
psycho.drop(columns=['employee_name'], inplace=True, errors='ignore')

for trait in ['O','C','E','A','N']:
    col = psycho[trait]
    psycho[f'{trait}_norm'] = (col - col.min()) / (col.max() - col.min())

psycho['psycho_ocean_risk_score'] = (
    psycho['N_norm'] *
    (1 - psycho['C_norm']) *
    (1 - psycho['A_norm']) *
    (1 - psycho['E_norm']) *
    psycho['O_norm']
)
psycho_feat = psycho[['user','psycho_ocean_risk_score']]

# ── LDAP — zscore du mois de départ par rapport aux autres utilisateurs ───────
ldap_path = f'{DATA_PATH}/LDAP'
ldap_all = []
for f_name in sorted(os.listdir(ldap_path)):
    df_tmp = pd.read_parquet(f'{ldap_path}/{f_name}')
    df_tmp['month'] = f_name.replace('.parquet','')
    ldap_all.append(df_tmp)
ldap = pd.concat(ldap_all, ignore_index=True)
ldap['month'] = pd.to_datetime(ldap['month'], errors='coerce')

user_tenure = ldap.groupby('user_id').agg(
    ldap_start = ('month','min'),
    ldap_end   = ('month','max')
).reset_index()
user_tenure['ldap_has_left']    = user_tenure['ldap_end'] < pd.Timestamp('2011-05-01')
user_tenure['ldap_tenure_months'] = (
    (user_tenure['ldap_end'] - user_tenure['ldap_start']).dt.days / 30
).round(1)

# Z-score de la tenure par rapport à la population (relatif)
mu, sigma = user_tenure['ldap_tenure_months'].mean(), user_tenure['ldap_tenure_months'].std()
user_tenure['ldap_zscore_departure'] = (
    (user_tenure['ldap_tenure_months'] - mu) / max(sigma, 1)
)
user_tenure.rename(columns={'user_id':'user'}, inplace=True)
ldap_feat = user_tenure[['user','ldap_zscore_departure','ldap_has_left']]

print('Psycho & LDAP OK')

In [ ]:
# ── Feature hebdomadaire LDAP : semaines avant le départ ─────────────────────
# Pour chaque (user, semaine), calcule le nombre de semaines restantes avant end_date.
# Négatif = déjà parti. Utile comme signal temporel pour les modèles séquentiels.

end_dates_series = weekly_grid['user'].map(user_end_date)
weekly_grid['ldap_weeks_to_departure'] = (
    (end_dates_series - weekly_grid['week_start']).dt.days / 7
).round(1)
# is_post_departure : utilisateur encore actif dans le système après son end_date
weekly_grid['ldap_is_post_departure'] = (weekly_grid['ldap_weeks_to_departure'] < 0).astype(int)

print('Feature LDAP hebdomadaire OK')

### Phase 4 — Assemblage final du dataset hebdomadaire

In [ ]:
# ── Colonnes à conserver par source ──────────────────────────────────────────

EMAIL_COLS = [
    'email_external_ratio_w','email_suspicious_ratio_w','email_bcc_ratio_w',
    'email_ext_att_ratio_w','email_off_hrs_ratio_w','email_not_main_pc_ratio_w',
    'email_cumul_avg_per_week','email_cumul_avg_size_sent',
    'email_zscore_nb_w','email_zscore_sent_w','email_zscore_ext_w','email_zscore_sus_w',
]

LOGON_COLS = [
    'logon_unclosed_ratio_w','logon_after_hrs_ratio_w','logon_multi_pc_flag_w',
    'logon_short_session_ratio_w','logon_not_main_pc_ratio_w',
    'session_duration_avg','session_duration_std',
    'logon_cumul_avg_session_h','logon_cumul_avg_per_week',
    'logon_zscore_nb_w','logon_zscore_after_hrs_w','logon_zscore_session_dur_w',
]

FILE_COLS = [
    'file_copy_ratio_w','file_write_ratio_w','file_delete_ratio_w',
    'file_copy_to_rem_ratio_w','file_from_rem_ratio_w',
    'file_events_per_file_w','file_off_hrs_ratio_w','file_suspicious_ratio_w',
    'file_open_then_copy_ratio_w','file_copy_then_del_ratio_w','file_write_then_del_ratio_w',
    'file_not_main_pc_ratio_w','decoy_cumul_access',
    'file_cumul_avg_per_week',
    'file_zscore_nb_w','file_zscore_copy_w','file_zscore_rem_w',
]

DEVICE_COLS = [
    'con_duration_avg','con_duration_std',
    'device_after_hrs_ratio_w','device_short_conn_ratio_w','device_not_main_pc_ratio_w',
    'device_cumul_avg_per_week','device_cumul_avg_duration_h',
    'device_zscore_nb_w','device_zscore_after_hrs_w',
]

HTTP_COLS = [
    'http_upload_ratio_w','http_download_ratio_w','http_off_hrs_ratio_w',
    'http_sus_domain_ratio_w','http_job_search_score_w',
    'http_avg_req_per_day_w','http_unique_domains_w','http_kw_flags_w',
    'http_cumul_avg_per_week',
    'http_zscore_nb_w','http_zscore_upload_w','http_zscore_kw_w',
]

STATIC_COLS = ['psycho_ocean_risk_score','ldap_zscore_departure','ldap_has_left']

# ── Merge séquentiel ──────────────────────────────────────────────────────────
feat = weekly_grid[['user','week_start','ldap_weeks_to_departure','ldap_is_post_departure']].copy()

def _merge(base, df, cols, on=['user','week_start']):
    keep_cols = on + [c for c in cols if c in df.columns]
    return base.merge(df[keep_cols], on=on, how='left')

feat = _merge(feat, email_w,  EMAIL_COLS)
feat = _merge(feat, logon_w,  LOGON_COLS)
feat = _merge(feat, file_w,   FILE_COLS)
feat = _merge(feat, device_w, DEVICE_COLS)
feat = _merge(feat, http_w,   HTTP_COLS)
feat = feat.merge(psycho_feat, on='user', how='left')
feat = feat.merge(ldap_feat[['user','ldap_zscore_departure','ldap_has_left']], on='user', how='left')

# ── Features cross-source ────────────────────────────────────────────────────
# Activité hors PC principal (agrégation sur les 4 sources)
for src_col in ['email_not_main_pc_ratio_w','logon_not_main_pc_ratio_w',
                'file_not_main_pc_ratio_w','device_not_main_pc_ratio_w']:
    if src_col not in feat.columns:
        feat[src_col] = 0.0

feat['users_off_main_pc_ratio_w'] = (
    feat['email_not_main_pc_ratio_w'] +
    feat['logon_not_main_pc_ratio_w'] +
    feat['file_not_main_pc_ratio_w']  +
    feat['device_not_main_pc_ratio_w']
) / 4

# Activité totale cette semaine (toutes sources)
for nb_col in ['nb_emails','nb_logon','nb_events','nb_conn','nb_total']:
    for src in [email_w, logon_w, file_w, device_w, http_w]:
        if nb_col in src.columns:
            feat = feat.merge(src[['user','week_start',nb_col]], on=['user','week_start'], how='left', suffixes=('','_dup'))

# Indicateur de semaine post-départ avec activité (signal d'accès non autorisé)
nb_cols = [c for c in feat.columns if c.startswith('nb_')]
if nb_cols:
    feat['users_total_activity_w'] = feat[nb_cols].fillna(0).sum(axis=1)
    feat['users_post_dep_activity_w'] = feat['ldap_is_post_departure'] * feat['users_total_activity_w']

# ── Nettoyage final ───────────────────────────────────────────────────────────
dup_cols = [c for c in feat.columns if c.endswith('_dup')]
feat.drop(columns=dup_cols, inplace=True)
feat.fillna(0, inplace=True)
feat.replace([np.inf, -np.inf], 0, inplace=True)

feat = feat.sort_values(['user','week_start']).reset_index(drop=True)
print(f'Dataset final : {feat.shape}  ({feat.shape[1]} features)')
feat.head()

In [ ]:
# ── Validation rapide : vérification no-leakage ───────────────────────────────
# Pour un utilisateur donné, le z-score de la semaine 1 doit être 0 (pas encore d'historique)
sample_user = feat['user'].iloc[0]
user_data   = feat[feat['user'] == sample_user].sort_values('week_start')

print(f'Utilisateur {sample_user} — premières semaines :')
print(user_data[['week_start','email_zscore_nb_w','logon_zscore_nb_w','file_zscore_nb_w']].head(6))

# Les 3 premières semaines doivent avoir zscore=0 (MIN_HIST_WEEKS=3 → pas assez d'historique)
assert (user_data['email_zscore_nb_w'].iloc[:MIN_HIST_WEEKS] == 0).all(), \
    'ALERTE : fuite détectée (z-score non nul sans historique suffisant)'
print(f'\n✓ No-leakage validé (z-scores nuls les {MIN_HIST_WEEKS} premières semaines)')

# Distributions de z-scores
zscore_cols = [c for c in feat.columns if 'zscore' in c]
print(f'\n{len(zscore_cols)} features z-score :')
print(feat[zscore_cols].describe().round(2))

In [ ]:
# ── Sauvegarde ────────────────────────────────────────────────────────────────
# Format parquet avec partition par semaine pour des lectures efficaces
feat.to_parquet(OUTPUT_PATH, index=False, partition_cols=['week_start'])
print(f'Sauvegardé : {OUTPUT_PATH}')
print(f'Dimensions : {feat.shape[0]:,} lignes × {feat.shape[1]} colonnes')
print(f'Semaines   : {feat["week_start"].nunique()} ({feat["week_start"].min().date()} → {feat["week_start"].max().date()})')
print(f'Utilisateurs : {feat["user"].nunique()}')
print(f'\nColonnes :\n{feat.columns.tolist()}')

# Export CSV optionnel pour inspection
# feat.to_csv('features_weekly.csv', index=False)